# GPUPerf-3B: All-In-One Benchmark & Evaluation Suite
## 🚀 Compute All Metrics, Tables, and Silicon Speedups in One Pass

This turnkey notebook runs the complete evaluation protocol across **Base Qwen** vs. **GPUPerf-3B (Final DPO)**:
1. **Language & Domain Metrics:** Cross-Entropy Loss & Perplexity (PPL) on held-out hardware kernels.
2. **DPO Preference Alignment:** Preference Accuracy (%) & Log-Probability Margin (+nats) over hardware pitfalls.
3. **Closed-Loop 3-Tier Gate:** Compilation Rate (%), `torch.matmul` Numerical Parity (%), and Physical Speedup via CUDA Events.
4. **Side-by-Side Generation:** Direct A/B output comparison on the same profiler report.
5. **Official Publication Benchmark Table:** Automatically formatted for README / Research Paper.

In [ ]:
# 1. Environment & Hardware Verification
import torch
!nvidia-smi

In [ ]:
# 2. Fast Install Dependencies
%%capture
!pip install --no-deps "xformers<0.0.29" triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. Clone Repository & Setup Dataset
import os
if not os.path.exists("GPUPerf"):;
    !git clone https://github.com/sarimahsan/GPUPerf.git
%cd GPUPerf
!git pull
!ls -lh ./data

In [ ]:
# 4. Unzip Checkpoint (If uploaded as zip)
# If you uploaded gpuperf_3b_dpo_final.zip to the root, unzip it:
if os.path.exists("../gpuperf_3b_dpo_final.zip"):
    !mkdir -p ./checkpoints/gpuperf_3b_dpo_final
    !unzip -o ../gpuperf_3b_dpo_final.zip -d ./checkpoints/gpuperf_3b_dpo_final
elif os.path.exists("./gpuperf_3b_dpo_final.zip"):
    !mkdir -p ./checkpoints/gpuperf_3b_dpo_final
    !unzip -o ./gpuperf_3b_dpo_final.zip -d ./checkpoints/gpuperf_3b_dpo_final

!ls -lh ./checkpoints/gpuperf_3b_dpo_final

In [ ]:
# 5. RUN THE COMPLETE ALL-IN-ONE EVALUATION SUITE
# Computes PPL, DPO Accuracy, 3-Tier Gate, and generates publication tables in one command!
!python scripts/evaluate_checkpoints.py \
    --checkpoint ./checkpoints/gpuperf_3b_dpo_final \
    --sft_data ./data/sft_multi_gpu.jsonl \
    --dpo_data ./data/dpo_multi_gpu.jsonl \
    --samples 15 \
    --output_json ./eval_summary.json

In [ ]:
# 6. Display Saved Summary JSON
import json
with open("./eval_summary.json", "r") as f:
    summary = json.load(f)

print(json.dumps(summary["metrics"], indent=2))